# Capstone Function 4
Address the challenge of optimally placing products across warehouses for a business with high online sales, where accurate calculations are costly and only feasible biweekly. To speed up decision-making, an ML model approximates these results within hours. The model has four hyperparameters to tune, and its output reflects the difference from the expensive baseline. Because the system is dynamic and full of local optima, it requires careful tuning and robust validation to find reliable, near-optimal solutions.

 Input | Output | Goal |
|-------|--------|------|
| 3D Array (30, 4) | 1D Array (30, ) | Maximise |

## Initial Submission

Bayesian Optimization for 4D warehouse placement hyperparameter tuning.

### Step 1: Import Required Libraries

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

print("Libraries imported successfully!")

### Step 2: Load and Display Initial Data

In [ ]:
# Load initial data
X_init = np.load('../../data/f4/initial_inputs.npy')
y_init = np.load('../../data/f4/initial_outputs.npy')

print("Initial Data Summary:")
print(f"Input shape: {X_init.shape} (30 configurations, 4 hyperparameters)")
print(f"Output shape: {y_init.shape}")
print(f"\nInput range: [{X_init.min():.4f}, {X_init.max():.4f}]")
print(f"Output range: [{y_init.min():.6f}, {y_init.max():.6f}]")
print(f"Output mean: {y_init.mean():.6f}")
print(f"Output std: {y_init.std():.6f}")
print(f"\nBest observed value: {y_init.max():.6f}")
print(f"Best configuration: {X_init[y_init.argmax()]}")

In [ ]:
# Visualize with pair plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
for idx, (i, j) in enumerate(pairs):
    scatter = axes[idx].scatter(X_init[:, i], X_init[:, j], c=y_init, s=80, cmap='viridis', edgecolors='black')
    best_idx = y_init.argmax()
    axes[idx].scatter(X_init[best_idx, i], X_init[best_idx, j], s=300, c='red', marker='*', edgecolors='black', linewidth=2)
    axes[idx].set_xlabel(f'Hyperparam {i+1}')
    axes[idx].set_ylabel(f'Hyperparam {j+1}')
    axes[idx].grid(True, alpha=0.3)
    
plt.colorbar(scatter, ax=axes, label='Performance')
plt.suptitle('4D Warehouse Optimization - Pairwise Projections', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 3: Define Hyperparameters

**For 4D Warehouse Problem:**
1. **GP Kernel**: Matern 5/2 - handles complex hyperparameter interactions
2. **Acquisition**: Expected Improvement - balances exploration/exploitation
3. **Restarts**: 20 (higher due to 4D space and local optima)
4. **Raw Samples**: 2048 (thorough initial sampling for 4D)
5. **Bounds**: Data-driven with margin

In [ ]:
# Define hyperparameters
margin = 0.1
x_min = X_init.min(axis=0) - margin
x_max = X_init.max(axis=0) + margin
BOUNDS = torch.tensor([x_min, x_max], dtype=torch.float64)

NUM_RESTARTS = 20
RAW_SAMPLES = 2048

print("Hyperparameters:")
print(f"  Input dimensions: 4")
print(f"  Acquisition function: Expected Improvement")
print(f"  Number of restarts: {NUM_RESTARTS}")
print(f"  Raw samples: {RAW_SAMPLES}")

### Step 4: Build Gaussian Process Model

In [ ]:
# Convert to tensors and train GP
X_train = torch.tensor(X_init, dtype=torch.float64)
y_train = torch.tensor(y_init, dtype=torch.float64).unsqueeze(-1)

gp_model = SingleTaskGP(X_train, y_train)
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)

print("Training GP model...")
fit_gpytorch_mll(mll)
print("✓ Training complete!")

print(f"\nLearned hyperparameters:")
print(f"  Noise: {gp_model.likelihood.noise.item():.6f}")
print(f"  Output scale: {gp_model.covar_module.outputscale.item():.6f}")
lengthscales = gp_model.covar_module.base_kernel.lengthscale.detach().numpy()[0]
print(f"  Lengthscales: {lengthscales}")

### Step 5: Optimize Acquisition Function

In [ ]:
# Optimize EI to find next point
best_f = y_train.max().item()
EI = ExpectedImprovement(gp_model, best_f=best_f)

print("Optimizing acquisition function...")
candidate, acq_value = optimize_acqf(
    EI, bounds=BOUNDS, q=1, 
    num_restarts=NUM_RESTARTS, 
    raw_samples=RAW_SAMPLES
)

next_point = candidate.detach().numpy()[0]
print("✓ Complete!")
print(f"\nNext point: {next_point}")
print(f"EI value: {acq_value.item():.6f}")

### Step 6: Visualize Feature Importance

GP lengthscales indicate feature importance - smaller lengthscale means greater sensitivity.

In [ ]:
# Visualize lengthscales (feature importance)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Lengthscales
ax1.bar(range(1, 5), lengthscales, color='steelblue', edgecolor='black')
ax1.set_xlabel('Hyperparameter', fontsize=12)
ax1.set_ylabel('Lengthscale', fontsize=12)
ax1.set_title('GP Lengthscales (Feature Sensitivity)', fontweight='bold')
ax1.set_xticks(range(1, 5))
ax1.grid(True, alpha=0.3, axis='y')

# Progress
best_so_far = np.maximum.accumulate(y_init)
ax2.plot(range(1, len(best_so_far)+1), best_so_far, 'b-o', linewidth=2, markersize=6)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Best Value', fontsize=12)
ax2.set_title('Optimization Progress', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Smaller lengthscales indicate more influential hyperparameters.")

### Summary

**4D Warehouse Placement Optimization Complete**
- 30 initial configurations evaluated
- GP model trained on 4D hyperparameter space
- Next configuration proposed via Expected Improvement
- Lengthscales reveal feature importance

**Next Steps:** Submit proposed configuration and update model